<div style="text-align: center; font-family: 'Times New Roman', Times, serif;">

## Cubli Simulation

Coded by Benjamin Saunders

Based on *The Cubli: Modeling and Nonlinear Attitude Control Utilizing Quaternions* by Bobrow et. al.
</div>

<div style="text-align: justify; font-family: 'Times New Roman', Times, serif;">

### Dynamics EOMs

According to Eq. 77 the dynamics are given as
$$
\begin{equation}\tag{77}
\left\{
\begin{array}{lcl}
\dot{q} & = & \frac{1}{2}G^T(q)\vec{\omega}_c \\
\dot{\vec{\theta}}_w & = & \vec{\omega}_w \\
\dot{\vec{\omega}}_c & = & \bar{I}_c^{-1}(-\vec{\omega}_c\times(\bar{I}_c\vec{\omega}_c + \bar{I}_w\vec{\omega}_w))-\bar{m}_cgl(G(q)\Gamma q) \\
& & \quad - b(G(q)\Lambda q(G(q)\Lambda q)^T)\vec{\omega}_c + \vec{\tau}_f(\vec{\omega}_w)-\vec{\tau}\\
\dot{\vec{\omega}}_w & = & \bar{I}_w^{-1}(-\vec{\tau}_f(\vec{\omega}_w)+\vec{\tau})
\end{array}
\right.
\end{equation}
$$
</div>

<div style="text-align: justify; font-family: 'Times New Roman', Times, serif;">

where, for reference, we define the quaternion product matrix, 
$$
\begin{equation}\tag{17}
G_{3\times 4}(q) = \begin{bmatrix}-q & q_wI_{3\times 3}-\tilde{q}\end{bmatrix} 
= 
\begin{bmatrix}
-q_x & -q_y & -q_z \\
q_w & -q_z & q_y \\
q_z & q_w & -q_x \\
-q_y & q_x & q_w \\
\end{bmatrix}^T
\end{equation}
$$

It is important to note that $q$ represents a standard Hamiltonian convention and represents a *local-to-global* rotation i.e. $q = q_\mathcal{G\leftarrow L}$.

And these three things,

$$
\begin{equation}\tag{73}
\bar{m}_c = m_s + 2m_w, \quad \Gamma_{4\times 4} 
= 
\begin{bmatrix}
1 & 1 & -1 & 0 \\ 
1 & -1 & 0 & 1 \\
-1 & 0 & -1 & 1 \\
0 & 1 & 1 & 1 \\
\end{bmatrix}
\end{equation}
$$

$$
\begin{equation}\tag{78}
\Lambda_{4\times 4} = \begin{bmatrix}
0 & 0 & 0 & 1 \\
0 & 0 & 1 & 0 \\
0 & -1 & 0 & 0 \\
-1 & 0 & 0 & 0 \\
\end{bmatrix}
\end{equation}
$$

And also the reaction wheel frictional torque,
$$
\begin{equation}\tag{76}
\tau_f(\omega_i) = \text{sign}(\omega_i)\left(\tau_c + b_w|\omega_i|+c_d|\omega_i|^2\right)
\end{equation}
$$

</div>

<div style="text-align: justify; font-family: 'Times New Roman', Times, serif;">

All those funky symbols are defined,

$$
\begin{aligned}
\vec{\theta}_w &= \text{Reaction wheels relative angular displacement vector } [\text{rad}] \\
\vec{\omega}_c &= \text{Cubli angular velocity vector } [\text{rad/s}] \\
\vec{\omega}_w &= \text{Composition of all three relative angular velocity} \\
&\quad \text{vectors of the reaction wheels } [\text{rad/s}] \\
\bar{I}_c &= \text{Cubli total inertia tensor on pivot point } O\text{, subtracting} \\
&\quad \text{the net inertia tensor } \bar{I}_w \text{ } [\text{kg} \cdot \text{m}^2] \\
\bar{I}_w &= \text{Net inertia tensor of the three reaction wheels} \\
&\quad \text{around each of their individual rotational axes } [\text{kg} \cdot \text{m}^2] \\
\bar{m}_c &= \text{Modified mass parameter } [\text{kg}] \\
g &= \text{Acceleration of gravity } [\text{m/s}^2] \\
l &= \text{Side length of the Cubli structure } [\text{m}] \\
b &= \text{Viscous surface friction coefficient at the pivot point } [\text{N} \cdot \text{m} \cdot \text{s/rad}] \\
\vec{\tau} &= \text{Vector of torques from the motors applied} \\
&\quad \text{on each of the three reaction wheels } [\text{N} \cdot \text{m}] \\
\tau_c &= \text{Coulomb friction of the motors } [\text{N} \cdot \text{m}] \\
b_w &= \text{Viscous friction coefficient of the motors } [\text{N} \cdot \text{m} \cdot \text{s/rad}] \\
c_d &= \text{Aerodynamic drag coefficient of the } \\
&\quad \text{reaction wheels } [\text{N} \cdot \text{m} \cdot \text{s}^2/\text{rad}^2]
\end{aligned}
$$

</div>

<div style="text-align: justify; font-family: 'Times New Roman', Times, serif;">

### Physics Code

</div>

In [ ]:
import numpy as np

class CubliPhysics:
    def __init__(self, params):
        """
        Initialize the Cubli physics model with dictionary of parameters.
        """
        # Inertia matrices
        self.I_c = params['I_c'] # Cube structure
        self.I_w = params['I_w'] # Reaction wheels
        
        # Precompute inverses for the ODEs
        self.inv_I_c = np.linalg.inv(self.I_c)
        self.inv_I_w = np.linalg.inv(self.I_w)
        
        # Mass and gravity parameters
        self.m_c = params['m_c']
        self.g   = params['g']
        self.l   = params['l']
        
        # Friction parameters
        self.b     = params['b']      # Surface viscous friction
        self.tau_c = params['tau_c']  # Motor Coulomb friction
        self.b_w   = params['b_w']    # Motor viscous friction
        self.c_d   = params['c_d']    # Motor aerodynamic drag
        
        # Constant Matrices
        self.Gamma = np.array([
            [ 1,  1, -1,  0],
            [ 1, -1,  0,  1],
            [-1,  0, -1,  1],
            [ 0,  1,  1,  1]
        ])
        
        self.Lambda = np.array([
            [ 0,  0,  0,  1],
            [ 0,  0, -1,  0],
            [ 0,  1,  0,  0],
            [-1,  0,  0,  0]
        ])

    def G_matrix(self, q):
        """Builds the 3x4 G(q) matrix for quaternion kinematics."""
        qw, qx, qy, qz = q
        return np.array([
            [-qx,  qw,  qz, -qy],
            [-qy, -qz,  qw,  qx],
            [-qz,  qy, -qx,  qw]
        ])

    def friction_torque(self, omega_w):
        """Calculates wheel friction torque (Eq 76)."""
        return np.sign(omega_w) * (self.tau_c + self.b_w * np.abs(omega_w) + self.c_d * omega_w**2)

    def ode_fun(self, t, x, tau, tau_d=np.zeros(3)):
        """
        The ODE function to be passed to an integrator.
        
        Parameters:
        t : float
            Time (required by solve_ivp, even if time-invariant)
        x : ndarray of shape (13,)
            State vector: [q (4), theta_w (3), omega_c (3), omega_w (3)]
        tau : ndarray of shape (3,)
            Control input: Motor torques for the 3 wheels, held constant (ZOH) for this step.
        tau_d : ndarray of shape (3,), optional
            Disturbance torque applied to the Cubli structure. 
            Expressed in the body-fixed coordinate frame!
            
        Returns:
        dx : ndarray of shape (13,)
            Time derivative of the state vector.
        """
        # Unpack state
        q       = x[0:4]
        theta_w = x[4:7]
        omega_c = x[7:10]
        omega_w = x[10:13]
        
        # Normalize quaternion to prevent numerical drift
        q = q / np.linalg.norm(q)
        
        # Quaternion kinematics
        G = self.G_matrix(q)
        dq = 0.5 * G.T @ omega_c
        
        # Kinematics of the wheels
        dtheta_w = omega_w
        
        # Gyroscopic terms
        H_total = self.I_c @ omega_c + self.I_w @ omega_w
        gyroscopic = np.cross(omega_c, H_total)
        
        # Gravity torque
        gravity = self.m_c * self.g * self.l * (G @ self.Gamma @ q)
        
        # Surface friction torque
        v = G @ self.Lambda @ q
        surface_friction = self.b * v * np.dot(v, omega_c) 
        
        # Motor friction torque
        tau_f = self.friction_torque(omega_w)
        
        # Angular acceleration of the Cubli
        domega_c = self.inv_I_c @ (
            -gyroscopic - gravity - surface_friction + tau_f - tau + tau_d
        )
        
        # Angular acceleration of the reaction wheels
        domega_w = self.inv_I_w @ (tau - tau_f)
        
        # Pack and return the derivatives
        return np.concatenate((dq, dtheta_w, domega_c, domega_w))

<div style="text-align: justify; font-family: 'Times New Roman', Times, serif;">

### Physics Code

</div>